In [2]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
 
LAST_NAME = "Endaya"
STUDENT_ID = "TUPM-26-4330"
 
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
 
print("Student:", LAST_NAME, STUDENT_ID)
print("Seed digit:", SEED_DIGIT)
print("ID sum:", ID_SUM)

Student: Endaya TUPM-26-4330
Seed digit: 0
ID sum: 18


In [3]:
# [CELL 2] CREATE WORKSPACE FOLDERS
# Define the folders (put the notebook inside Lab6_File_Handling_Additional_Exercises)
WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"        # raw Kaggle CSV goes here (NOT uploaded to GitHub)
OUTPUT_DIR = WORK_DIR / "output"    # generated files
BACKUP_DIR = WORK_DIR / "backup"    # backups
 
# Create the folders if they do not exist yet
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)
 
# Verify the workspace
print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

Workspace: C:\Users\WINDOWS\Lab6_File_Handling_Additional_Exercise
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [4]:
# [CELL 3] EXERCISE 1 PARAMETERS
# Built-in modules only (no Pandas or NumPy)
import csv
import json
import shutil
from collections import Counter
 
# Dataset location (download weblog.csv from Kaggle into the data folder)
dataset_path = DATA_DIR / "weblog.csv"
 
# Student-specific rules
records_to_process = 1000 + ID_SUM          # first 1000 + ID_SUM data records
target_class = (SEED_DIGIT % 4) + 2         # 2, 3, 4 or 5 -> 2xx, 3xx, 4xx, 5xx
class_label = f"{target_class}xx"
 
print("Dataset:", dataset_path)
print("Records to process:", records_to_process)
print("Target status class:", class_label)

Dataset: data\weblog.csv
Records to process: 1018
Target status class: 2xx


In [5]:
# [CELL 4] READ THE LOG LINE BY LINE AND FILTER
# Name of the status column (the dataset spells it "Staus")
status_col = "Staus"
 
# Containers for the results
selected = []              # records in the target class (original order kept)
status_counts = Counter()  # requests per status code
url_counts = Counter()     # requests per URL (target class only)
processed = 0              # data records read
invalid = 0                # records with a missing/invalid status
 
if not dataset_path.exists():
    print("Dataset not found:", dataset_path)
else:
    with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
        # DictReader yields one row at a time, so the whole file is never loaded
        reader = csv.DictReader(file)
        if "Staus" not in reader.fieldnames and "Status" in reader.fieldnames:
            status_col = "Status"
 
        for row in reader:
            # Stop after the required number of records
            if processed >= records_to_process:
                break
            processed += 1
 
            # Convert the status to an integer; skip bad values
            try:
                status = int(row[status_col])
            except (KeyError, TypeError, ValueError):
                invalid += 1
                continue
 
            # Keep the record only if it belongs to the target class (e.g. 404 // 100 = 4)
            if status // 100 == target_class:
                url = (row.get("URL") or "").strip()
                selected.append({
                    "IP": row.get("IP", ""),
                    "Time": row.get("Time", ""),
                    "URL": url,
                    status_col: status,
                })
                status_counts[status] += 1
                url_counts[url] += 1
 
# Five most requested URLs (ties keep the order they first appeared)
top_urls = url_counts.most_common(5)
 
print("Records processed:", processed)
print("Invalid records skipped:", invalid)
print("Records in target class:", len(selected))
print("Count per status code:", dict(sorted(status_counts.items())))
print("Top 5 URLs:")
for url, count in top_urls:
    print(" ", count, url)

Dataset not found: data\weblog.csv
Records processed: 0
Invalid records skipped: 0
Records in target class: 0
Count per status code: {}
Top 5 URLs:


In [6]:
# CELL 5 EXPORT FILTERED CSV
# Path of the filtered CSV
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"
 
# Write the selected records (a header-only file is written if there are none)
with open(filtered_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["IP", "Time", "URL", status_col])
    writer.writeheader()
    writer.writerows(selected)
 
print("Created:", filtered_path)
print("Rows written:", len(selected))

Created: output\Endaya_TUPM-26-4330_weblog_filtered.csv
Rows written: 0


In [7]:
# CELL 6 WRITE JSON SUMMARY
# Path of the JSON summary
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"
 
# Summary dictionary
summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": processed,
    "invalid_records": invalid,
    "target_status_class": class_label,
    "records_in_target_class": len(selected),
    "count_per_status_code": {str(k): v for k, v in sorted(status_counts.items())},
    "top_five_urls": [{"url": u, "requests": c} for u, c in top_urls],
    "message": "No records found in target class" if not selected else "OK",
}
 
# Save as readable JSON
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)
 
print("Created:", json_path)

Created: output\Endaya_TUPM-26-4330_weblog_summary.json


In [8]:
# CELL 7 BACKUP AND VERIFY THE JSON FILE
# Copy the JSON file (copy2 keeps the metadata)
backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)
 
# Verify the backup exists and matches the original size
backup_exists = backup_path.exists()
same_size = backup_exists and backup_path.stat().st_size == json_path.stat().st_size
backup_status = "VERIFIED" if same_size else "FAILED"
 
print("Backup:", backup_path)
print("Backup exists:", backup_exists)
print("Backup verification:", backup_status)

Backup: backup\Endaya_TUPM-26-4330_weblog_summary.json
Backup exists: True
Backup verification: VERIFIED


In [9]:
# CELL 8 FINAL OUTPUT (ASSESSMENT DATA)
print("Records processed:", processed)
print("Target status class:", class_label)
print("Records in target class:", len(selected))
print("Count per status code:", dict(sorted(status_counts.items())))
print("Five most requested URLs:", top_urls)
print("Filtered CSV file:", filtered_path.name)
print("JSON summary file:", json_path.name)
print("Backup verification:", backup_status)

Records processed: 0
Target status class: 2xx
Records in target class: 0
Count per status code: {}
Five most requested URLs: []
Filtered CSV file: Endaya_TUPM-26-4330_weblog_filtered.csv
JSON summary file: Endaya_TUPM-26-4330_weblog_summary.json
Backup verification: VERIFIED
